## Cell 1 — Notebook Goal and Flow

This notebook demonstrates a full mini-ingestion pipeline from **Random User API** to **MySQL**.

### What this notebook does
1. Reads environment-based configuration
2. Connects to MySQL through SQLAlchemy
3. Calls API with retries (resilient API access)
4. Normalizes nested JSON into a table-friendly DataFrame
5. Applies basic data quality checks
6. Creates a target table if needed
7. Performs idempotent upsert using `user_uuid`
8. Verifies the load with SQL queries

> Run cells in order from top to bottom because each later cell depends on objects created earlier.

In [13]:
# Standard library
import logging
import os
from pathlib import Path
from typing import Any, Dict, Optional

# Third-party libraries
import pandas as pd
import requests
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from sqlalchemy import create_engine, text

# =========================
# Logging Configuration
# =========================
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / "api_to_mysql_ingestion.log"

logger = logging.getLogger("api_to_mysql_ingestion")
logger.setLevel(logging.INFO)
logger.propagate = False

if not logger.handlers:
    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s")
    file_handler = logging.FileHandler(LOG_FILE, encoding="utf-8")
    file_handler.setFormatter(formatter)
    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    logger.addHandler(stream_handler)

# =========================
# API Configuration
# =========================
API_BASE_URL = os.getenv("API_BASE_URL", "https://randomuser.me")
RESULTS_PER_CALL = int(os.getenv("RESULTS_PER_CALL", "100"))

# =========================
# MySQL Configuration
# =========================
MYSQL_HOST = os.getenv("MYSQL_HOST", "db")
MYSQL_PORT = int(os.getenv("MYSQL_PORT", "3306"))
MYSQL_DATABASE = os.getenv("MYSQL_DATABASE", "demo")
MYSQL_USER = os.getenv("MYSQL_USER", "demo_user")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD", "demo_pass")

logger.info("Configuration loaded | API_BASE_URL=%s | RESULTS_PER_CALL=%s | MYSQL=%s@%s:%s/%s", API_BASE_URL, RESULTS_PER_CALL, MYSQL_USER, MYSQL_HOST, MYSQL_PORT, MYSQL_DATABASE)
print("API_BASE_URL:", API_BASE_URL)
print("RESULTS_PER_CALL:", RESULTS_PER_CALL)
print("MYSQL:", f"{MYSQL_USER}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}")
print("LOG_FILE:", str(LOG_FILE))

2026-03-01 17:13:02,610 | INFO | api_to_mysql_ingestion | Configuration loaded | API_BASE_URL=https://randomuser.me | RESULTS_PER_CALL=100 | MYSQL=demo_user@db:3306/demo


API_BASE_URL: https://randomuser.me
RESULTS_PER_CALL: 100
MYSQL: demo_user@db:3306/demo
LOG_FILE: logs/api_to_mysql_ingestion.log


## Cell 2 — Create MySQL Engine

This cell builds the SQLAlchemy connection URL and validates that MySQL is reachable.

In [14]:
# SQLAlchemy URL format for MySQL + PyMySQL driver.
mysql_url = (
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}"
    f"@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
)

try:
    # Create an engine once and re-use it in later cells.
    # pool_pre_ping=True helps recover from stale DB connections.
    engine = create_engine(mysql_url, pool_pre_ping=True)

    # Fast connectivity check so we fail early if DB is unreachable.
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))

    logger.info("Connected to MySQL successfully")
    print("✅ Connected to MySQL")
except Exception as exc:
    # Log full traceback to file + console, then raise a clearer notebook error.
    logger.exception("MySQL connection failed")
    raise RuntimeError("Failed to connect to MySQL. Check DB credentials/host and container health.") from exc

2026-03-01 17:13:13,429 | INFO | api_to_mysql_ingestion | Connected to MySQL successfully


✅ Connected to MySQL


## Cell 3 — API Helper Functions with Retry

This section defines reusable API utilities:
- request headers builder
- custom API error type
- `get_json()` with retry/backoff and response validation

In [15]:
# Custom exception used to trigger retries for controlled API failures.
class ApiError(RuntimeError):
    pass

# Common headers for API calls.
def build_headers():
    return {"Accept": "application/json"}

# Retry policy is applied to network failures and ApiError.
@retry(
    reraise=True,
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=1, max=10),
    retry=retry_if_exception_type((requests.RequestException, ApiError)),
)
def get_json(path: str, params=None):
    # Build a safe URL from base + path (handles leading/trailing slashes).
    base = (os.getenv("API_BASE_URL") or API_BASE_URL or "https://randomuser.me").rstrip("/")
    endpoint = (path or "").lstrip("/")
    url = f"{base}/{endpoint}" if endpoint else f"{base}/"

    # Guardrail: ensure scheme is present to avoid MissingSchema errors.
    if not base.startswith(("http://", "https://")):
        raise ApiError(f"Invalid API_BASE_URL (missing scheme): {base!r}")

    logger.info("Calling API | url=%s | params=%s", url, params)

    try:
        # Main HTTP call
        resp = requests.get(url, headers=build_headers(), params=params, timeout=30)
    except requests.RequestException:
        # Retry will trigger because this exception type is configured above.
        logger.exception("Network failure calling API")
        raise

    # Treat HTTP 4xx/5xx as controlled failures with context.
    if resp.status_code >= 400:
        raise ApiError(f"API error {resp.status_code}. URL={resp.url}. Body={resp.text[:300]}")

    # Validate response type before JSON parsing.
    ct = (resp.headers.get("Content-Type") or "").lower()
    if "json" not in ct:
        raise ApiError(f"Non-JSON response. URL={resp.url}. Content-Type={ct}. Body={resp.text[:300]}")

    try:
        payload = resp.json()
    except ValueError as exc:
        raise ApiError(f"JSON decode failed. URL={resp.url}. Body={resp.text[:300]}") from exc

    logger.info("API call successful | status=%s | url=%s", resp.status_code, resp.url)
    return payload

In [16]:
# Optional diagnostic cell: confirms API endpoint behavior independently from helper function.
import os, requests

print("API_BASE_URL =", os.getenv("API_BASE_URL"))

url = "https://randomuser.me/api/?results=1"
r = requests.get(url, timeout=30)

print("Status:", r.status_code)
print("Final URL:", r.url)  # useful to detect redirects
print("Content-Type:", r.headers.get("Content-Type"))
print("First 300 chars:\n", r.text[:300])

API_BASE_URL = https://randomuser.me
Status: 200
Final URL: https://randomuser.me/api/?results=1
Content-Type: application/json; charset=utf-8
First 300 chars:
 {"results":[{"gender":"female","name":{"title":"Miss","first":"Harper","last":"Gill"},"location":{"street":{"number":7886,"name":"Grand Marais Ave"},"city":"Port Elgin","state":"Québec","country":"Canada","postcode":"P9B 9L3","coordinates":{"latitude":"-69.7147","longitude":"43.0532"},"timezone":{"o


## Cell 4 — Extract Data from API

This cell requests a batch of records from Random User and stores the raw JSON payload.

In [17]:
# Random User API supports batching with the `results` query parameter.
# Using '/api/' assumes API_BASE_URL does not already include '/api'.
try:
    # Extract one API batch and keep top-level payload for downstream cells.
    raw_response = get_json("/api/", params={"results": RESULTS_PER_CALL, "format": "json"})

    # Log useful metadata for troubleshooting/auditing.
    logger.info("Extracted payload from API | top_level_keys=%s", list(raw_response.keys()))

    # Show top-level keys interactively in notebook output.
    raw_response.keys()
except Exception:
    # Keep traceback in logs and stop pipeline execution in notebook.
    logger.exception("Extraction step failed")
    raise

2026-03-01 17:13:32,408 | INFO | api_to_mysql_ingestion | Calling API | url=https://randomuser.me/api/ | params={'results': 100, 'format': 'json'}
2026-03-01 17:13:33,560 | INFO | api_to_mysql_ingestion | API call successful | status=200 | url=https://randomuser.me/api/?results=100&format=json
2026-03-01 17:13:33,563 | INFO | api_to_mysql_ingestion | Extracted payload from API | top_level_keys=['results', 'info']


## Cell 5 — Normalize JSON to DataFrame

Converts nested JSON user objects into a flat tabular structure for transformation and loading.

In [18]:
# Pull array of user records from payload.
users = raw_response.get("results", [])

# Flatten nested JSON fields into dot-notated columns (e.g., name.first, login.uuid).
df = pd.json_normalize(users)

# Preview normalized data.
df.head()

,gender,email,phone,cell,nat,name.title,name.first,name.last,location.street.number,location.street.name,...,login.sha256,dob.date,dob.age,registered.date,registered.age,id.name,id.value,picture.large,picture.medium,picture.thumbnail
0,male,homero.aleman@example.com,(693) 605 3060,(646) 583 2956,MX,Mr,Homero,Alemán,5138,Continuación Zamora,...,98e748fd20c4123e567931785342ee359d1dd1cfbbe3fa...,1990-04-14T23:49:39.252Z,35,2015-08-20T16:50:39.736Z,10,NSS,71 08 34 3658 6,https://randomuser.me/api/portraits/men/60.jpg,https://randomuser.me/api/portraits/med/men/60...,https://randomuser.me/api/portraits/thumb/men/...
1,male,bojan.danicic@example.com,038-2020-294,063-9612-622,RS,Mr,Bojan,Daničić,3243,Žitkovačka,...,995d9faf54dd07fd0419e18fc9a6ab17d6417fe13d9812...,1968-07-28T02:34:09.609Z,57,2010-10-23T13:13:32.683Z,15,SID,039493152,https://randomuser.me/api/portraits/men/82.jpg,https://randomuser.me/api/portraits/med/men/82...,https://randomuser.me/api/portraits/thumb/men/...
2,female,michelle.richards@example.com,0171 799 5497,07345 589877,GB,Mrs,Michelle,Richards,907,Green Lane,...,959a7ad8f3f897e36a625026b2c9beece94eecc017cb69...,1963-12-07T00:11:24.749Z,62,2016-11-04T13:42:47.117Z,9,NINO,CM 71 47 54 A,https://randomuser.me/api/portraits/women/79.jpg,https://randomuser.me/api/portraits/med/women/...,https://randomuser.me/api/portraits/thumb/wome...
3,male,roman.grabowski@example.com,0887-3295723,0175-1833667,DE,Mr,Roman,Grabowski,4869,Schulweg,...,4cf4567a1c10c1ac4a2f211dd77f2856574f97f94abb52...,1956-04-17T01:04:34.227Z,69,2013-06-17T10:18:05.289Z,12,SVNR,12 160456 G 398,https://randomuser.me/api/portraits/men/68.jpg,https://randomuser.me/api/portraits/med/men/68...,https://randomuser.me/api/portraits/thumb/men/...
4,male,carlo.martinez@example.com,075 234 45 69,077 901 27 29,CH,Monsieur,Carlo,Martinez,9388,Cours Charlemagne,...,ec860caf6ce0cbfb0cb3d798be75cd295cac47b05114bb...,1956-06-12T02:59:04.541Z,69,2008-01-14T05:38:02.551Z,18,AVS,756.4424.0936.12,https://randomuser.me/api/portraits/men/92.jpg,https://randomuser.me/api/portraits/med/men/92...,https://randomuser.me/api/portraits/thumb/men/...


## Cell 6 — Data Quality and Model Shaping

This cell performs data checks and transforms raw columns into a cleaner model for MySQL.

Checks included:
- dataset is not empty
- natural key `login.uuid` exists
- `user_uuid` has no nulls and no duplicates in this batch

In [19]:
import hashlib
import json
import numpy as np

try:
    # Basic ingestion guardrails before transformation.
    assert not df.empty, "API returned no data"
    assert "login.uuid" in df.columns, "Expected natural key field 'login.uuid'"

    # Select fields needed for relational model.
    df_model = df[[
        "login.uuid",
        "gender",
        "name.first",
        "name.last",
        "email",
        "dob.date",
        "dob.age",
        "location.country",
        "phone"
    ]].copy()

    # Rename source fields to target schema column names.
    df_model.columns = [
        "user_uuid",
        "gender",
        "first_name",
        "last_name",
        "email",
        "dob",
        "age",
        "country",
        "phone"
    ]

    # Convert DOB to date-only values for MySQL DATE column.
    df_model["dob"] = pd.to_datetime(df_model["dob"], utc=True, errors="coerce").dt.date

    # Key integrity checks.
    assert df_model["user_uuid"].notna().all(), "Missing user_uuid values"
    assert df_model["user_uuid"].is_unique, "Duplicate user_uuid values in this batch"

    # Hashing input fields (used to detect exact record duplicates).
    hash_columns = [
        "user_uuid", "gender", "first_name", "last_name",
        "email", "dob", "age", "country", "phone"
    ]

    def _normalize_for_hash(value):
        # Normalize nulls/dates into stable hash-friendly values.
        if pd.isna(value):
            return None
        if hasattr(value, "isoformat"):
            return value.isoformat()
        return value

    def _compute_record_hash(row):
        payload = {col: _normalize_for_hash(row[col]) for col in hash_columns}
        canonical_json = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
        return hashlib.sha256(canonical_json.encode("utf-8")).hexdigest()

    df_model["record_hash"] = df_model.apply(_compute_record_hash, axis=1)

    # Deduplicate identical records in this batch by hash.
    rows_before_dedup = len(df_model)
    df_model = df_model.drop_duplicates(subset=["record_hash"]).reset_index(drop=True)
    rows_after_dedup = len(df_model)
    duplicates_removed = rows_before_dedup - rows_after_dedup

    assert df_model["record_hash"].notna().all(), "Missing record_hash values after hash generation"
    assert df_model["record_hash"].is_unique, "record_hash must be unique after dedup"

    logger.info("Transform complete | rows_before=%s | rows_after=%s | duplicates_removed=%s", rows_before_dedup, rows_after_dedup, duplicates_removed)
    print(f"Rows before dedup: {rows_before_dedup}")
    print(f"Rows after dedup: {rows_after_dedup}")
    print(f"Duplicates removed: {duplicates_removed}")
    df_model.head()
except Exception:
    logger.exception("Transform/data-quality step failed")
    raise

2026-03-01 17:14:28,525 | INFO | api_to_mysql_ingestion | Transform complete | rows_before=100 | rows_after=100 | duplicates_removed=0


Rows before dedup: 100
Rows after dedup: 100
Duplicates removed: 0


## Cell 7 — Ensure Target Table Exists

Creates `random_users` table if missing so the load is repeatable across fresh environments.

In [20]:
# Idempotent DDL: safe to run multiple times.
create_sql = """
CREATE TABLE IF NOT EXISTS random_users (
  user_uuid VARCHAR(36) PRIMARY KEY,
  record_hash CHAR(64) NOT NULL,
  gender VARCHAR(10),
  first_name VARCHAR(100),
  last_name VARCHAR(100),
  email VARCHAR(255),
  dob DATE,
  age INT,
  country VARCHAR(100),
  phone VARCHAR(50),
  ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  KEY idx_random_users_record_hash (record_hash)
);
"""

try:
    # Ensure target table exists before load.
    with engine.begin() as conn:
        conn.execute(text(create_sql))

    logger.info("Ensured table exists: random_users")
    print("✅ Ensured table exists: random_users")
except Exception:
    logger.exception("DDL step failed for table random_users")
    raise

2026-03-01 17:14:34,268 | INFO | api_to_mysql_ingestion | Ensured table exists: random_users


✅ Ensured table exists: random_users


## Cell 8 — Upsert into MySQL

This cell loads data with MySQL upsert semantics (`ON DUPLICATE KEY UPDATE`) so re-runs do not create duplicate primary keys.

In [21]:
from sqlalchemy import MetaData, Table
from sqlalchemy.dialects.mysql import insert as mysql_insert
import numpy as np

try:
    # Defensive copy before load-time tweaks.
    df_model = df_model.copy()

    # Optional quick preview of important load columns.
    print(df_model[["dob", "record_hash"]].head())

    # Reflect table metadata from the live database.
    metadata = MetaData()
    random_users = Table("random_users", metadata, autoload_with=engine)

    # Convert DataFrame rows into list[dict] for SQLAlchemy bulk insert.
    records = df_model.to_dict(orient="records")
    stmt = mysql_insert(random_users).values(records)

    # Upsert behavior keeps pipeline idempotent across reruns.
    upsert_stmt = stmt.on_duplicate_key_update(
        record_hash=stmt.inserted.record_hash,
        gender=stmt.inserted.gender,
        first_name=stmt.inserted.first_name,
        last_name=stmt.inserted.last_name,
        email=stmt.inserted.email,
        dob=stmt.inserted.dob,
        age=stmt.inserted.age,
        country=stmt.inserted.country,
        phone=stmt.inserted.phone
)

    # Execute load in one transaction.
    with engine.begin() as conn:
        conn.execute(upsert_stmt)

    logger.info("Load complete | upserted_rows=%s", len(records))
    print(f"✅ Upserted {len(records)} records into random_users")
except Exception:
    logger.exception("Load/upsert step failed")
    raise

2026-03-01 17:14:40,566 | INFO | api_to_mysql_ingestion | Load complete | upserted_rows=100


          dob                                        record_hash
0  1990-04-14  c6bf1825ba26fb7169029e5483133b78cad8fcffc9a75c...
1  1968-07-28  6e42a9bfddd42f03e6ff6918c4a86d3d92dee5cb6003ba...
2  1963-12-07  355f5bdee4a16b4ef013c748b2d1874b0a9c8a16dc74c9...
3  1956-04-17  3cb68ce411a6a57aec9504090152ca862a2d9d281b9e57...
4  1956-06-12  1eeaa8e770d791adcf8e91a8c3c39a2581a35638f501f4...
✅ Upserted 100 records into random_users


## Cell 9 — Verify Data Load

Runs post-load checks: total row count and latest sample rows.

In [22]:
# Query total rows and a recent sample to validate successful ingestion.
try:
    with engine.connect() as conn:
        rows = conn.execute(text("SELECT COUNT(*) AS c FROM random_users")).scalar_one()
        sample = conn.execute(text("SELECT * FROM random_users ORDER BY ingested_at DESC LIMIT 5")).mappings().all()

    # Log row count for quick run summary in the log file.
    logger.info("Verification complete | table=random_users | row_count=%s", rows)
    rows, sample
except Exception:
    logger.exception("Verification step failed")
    raise

2026-03-01 17:14:47,583 | INFO | api_to_mysql_ingestion | Verification complete | table=random_users | row_count=200


In [ ]:
# Optional scratch cell for ad-hoc testing.